In [1]:
import jax
import jax.numpy as jnp

from shapiq.game.table import TableGame
from shapiq.players import FinitePlayers

In [2]:
coalitions = jnp.array(
    [
        [0, 0, 0],  # empty
        [1, 0, 0],  # {0}
        [0, 1, 0],  # {1}
        [1, 1, 0],  # {0, 1}
        [0, 0, 1],  # {2}
        [1, 0, 1],  # {0, 2}
        [0, 1, 1],  # {1, 2}
        [1, 1, 1],  # {0, 1, 2}
    ],
    dtype=bool,
)
g = TableGame(jnp.array([[0.0, 1.0, 2.0, 4.0, 3.0, 5.0, 6.0, 10.0]]), FinitePlayers(3))
print("game batch:", g.shape, "shared players:", g.players.shape)
for coalition, value in zip(coalitions.tolist(), g(coalitions).tolist(), strict=True):
    print(coalition, "->", value)

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


game batch: (1,) shared players: ()
[False, False, False] -> 0.0
[True, False, False] -> 1.0
[False, True, False] -> 2.0
[True, True, False] -> 4.0
[False, False, True] -> 3.0
[True, False, True] -> 5.0
[False, True, True] -> 6.0
[True, True, True] -> 10.0


# Build the ExactComputer for Shapley

In [3]:
from shapiq.game.exact import ExactComputer
from shapiq.index import ShapleyValueIndex

computer = ExactComputer.from_game(g)

shapley_index = ShapleyValueIndex()
shapley_values = computer(shapley_index)
shapley_values

AdditiveMoebiusGame(values=Array([[2.333334 , 3.3333342, 4.3333344]], dtype=float32), players=FinitePlayers(ids=Array([0, 1, 2], dtype=int32), players=None, latent_n_players=3, n_players=3), interactions=Array([[ True, False, False],
       [False,  True, False],
       [False, False,  True]], dtype=bool), additivity=1)

In [4]:
from shapiq.game.exact import ExactComputer
from shapiq.index import ShapleyInteractionIndex

computer = ExactComputer.from_game(g)

shapley_index = ShapleyInteractionIndex(max_order=2)
shapley_values = computer(shapley_index)
shapley_values

MoebiusGame(values=Array([[2.333334 , 3.3333342, 1.5000007, 4.3333344, 1.5000007, 1.500001 ]],      dtype=float32), players=FinitePlayers(ids=Array([0, 1, 2], dtype=int32), players=None, latent_n_players=3, n_players=3), interactions=Array([[ True, False, False],
       [False,  True, False],
       [ True,  True, False],
       [False, False,  True],
       [ True, False,  True],
       [False,  True,  True]], dtype=bool), additivity=2)

# One cached table, many indices

`ExactComputer.from_game` evaluates the game once on all `2**n` coalitions and stores them as a `TableGame`. Every index computed afterwards reuses that table. The index decides which game the result is: semivalues (`ShapleyValueIndex`) give an `AdditiveMoebiusGame`, cardinal interaction indices a `MoebiusGame`.

In [5]:
from shapiq.index import (
    AlphaBetaIndex,
    BanzhafIndex,
    ShapleyInteractionIndex,
    ShapleyValueIndex,
    WeightedBanzhafIndex,
)

computer = ExactComputer.from_game(g)


def show(game):
    """Print each interaction set of the first game with its coefficient."""
    print(type(game).__name__, "batch", game.shape)
    for members, value in zip(game.interactions.tolist(), game.values[0].tolist(), strict=True):
        print("  ", {i for i, m in enumerate(members) if m}, f"{value:+.4f}")

## First-order indices: one value per player

`ShapleyValueIndex` needs no arguments and returns an `AdditiveMoebiusGame`. `BanzhafIndex(max_order=1)` and `WeightedBanzhafIndex(p=..., max_order=1)` return a 1-additive `MoebiusGame`.

In [6]:
show(computer(ShapleyValueIndex()))
show(computer(BanzhafIndex(max_order=1)))
show(computer(WeightedBanzhafIndex(p=0.3, max_order=1)))

AdditiveMoebiusGame batch (1,)
   {0} +2.3333
   {1} +3.3333
   {2} +4.3333
MoebiusGame batch (1,)
   {0} +2.2500
   {1} +3.2500
   {2} +4.2500
MoebiusGame batch (1,)
   {0} +1.6900
   {1} +2.6900
   {2} +3.6900


## Interaction indices: coefficients on sets up to `max_order`

SII and BII cover all sets from `min_order=1` to `max_order`. Reading these values as Möbius coefficients is an interpretation: for orders above one, the resulting game generally differs from `g`.

In [7]:
show(computer(ShapleyInteractionIndex(max_order=2)))
show(computer(BanzhafIndex(max_order=2)))

MoebiusGame batch (1,)
   {0} +2.3333
   {1} +3.3333
   {0, 1} +1.5000
   {2} +4.3333
   {0, 2} +1.5000
   {1, 2} +1.5000
MoebiusGame batch (1,)
   {0} +2.2500
   {1} +3.2500
   {0, 1} +1.5000
   {2} +4.2500
   {0, 2} +1.5000
   {1, 2} +1.5000


## Alpha-Beta interaction indices

`AlphaBetaIndex` generalizes Shapley interactions via Beta(`alpha`, `beta`) weights. Setting `alpha = 1` and `beta = 1` recovers `ShapleyInteractionIndex`.

In [8]:
show(computer(AlphaBetaIndex(alpha=4.0, beta=1.0, max_order=2)))

ab_1_1 = computer(AlphaBetaIndex(alpha=1.0, beta=1.0, max_order=2)).values
sii_2 = computer(ShapleyInteractionIndex(max_order=2)).values
print(
    "AlphaBeta(1, 1) equals SII:",
    bool(jnp.allclose(ab_1_1, sii_2, atol=1e-5)),
)

MoebiusGame batch (1,)
   {0} +1.4667
   {1} +2.4667
   {0, 1} +1.2000
   {2} +3.4667
   {0, 2} +1.2000
   {1, 2} +1.2000
AlphaBeta(1, 1) equals SII: True


## The result is a game

The Shapley game is additive: a coalition's value is the sum of its members' Shapley values. Its grand coalition therefore gets `v(N) - v(∅)` (efficiency).

In [9]:
shapley_game = computer(ShapleyValueIndex())
print("coalition values:", shapley_game(coalitions[:, None]).ravel().tolist())
print(
    "grand coalition:",
    float(shapley_game(jnp.ones(3, dtype=bool))[0]),
    "= v(N) - v(∅) =",
    float(g.values[0, -1] - g.values[0, 0]),
)

coalition values: [0.0, 2.333333969116211, 3.33333420753479, 5.666667938232422, 4.333334445953369, 6.66666841506958, 7.666668891906738, 10.000001907348633]
grand coalition: 10.000001907348633 = v(N) - v(∅) = 10.0


## Batches: every game with every weight

Game and index batches broadcast like arrays. To pair every game with every `p`, give the computer a trailing axis.

In [10]:
games = TableGame(
    jnp.array(
        [[0.0, 1.0, 2.0, 4.0, 3.0, 5.0, 6.0, 10.0], [0.0, 2.0, 1.0, 3.0, 1.0, 3.0, 2.0, 4.0]]
    ),
    FinitePlayers(3),
)
batched = ExactComputer.from_game(games)
p_values = jnp.array([0.3, 0.5, 0.7])

every_pair = batched[:, None](WeightedBanzhafIndex(p=p_values, max_order=1))
print("games x p:", every_pair.shape)
print(every_pair.values)

games x p: (2, 3)
[[[1.69       2.69       3.69      ]
  [2.25       3.25       4.25      ]
  [2.8899999  3.8899999  4.89      ]]

 [[1.9999999  0.99999994 1.        ]
  [2.         1.         1.        ]
  [2.         1.         0.99999994]]]


## Compiling the whole computation with `jax.jit`

Games, indices and computers are pytrees, so the full pipeline, from evaluating every coalition to building the result game, compiles into one function, and the result stays a game. Each index type compiles once; afterwards, new games or new `p` values with the same shapes reuse the compiled version. A different `n_players` or `max_order` changes the shapes and compiles again.

In [11]:
from time import perf_counter


@jax.jit
def compute(game, index):
    return ExactComputer.from_game(game)(index)


a = perf_counter()
shapley_games = compute(games, ShapleyValueIndex())  # compiles for ShapleyValueIndex
print(type(shapley_games).__name__, shapley_games.shape)
print(shapley_games.values)
print(f"Time taken: {perf_counter() - a:.2f} seconds")

b = perf_counter()
doubled = TableGame(2 * games.values, FinitePlayers(3))
print("doubled game:", compute(doubled, ShapleyValueIndex()).values)  # reuses it
print(f"Time taken: {perf_counter() - b:.2f} seconds")


for p in (0.3, 0.7):  # compiles once for WeightedBanzhafIndex, then reuses it
    c = perf_counter()
    banzhaf = compute(games, WeightedBanzhafIndex(p=p, max_order=1))
    print(f"p = {p}:", banzhaf.values)
    print(f"Time taken: {perf_counter() - c:.2f} seconds")

AdditiveMoebiusGame (2,)
[[2.333334  3.3333342 4.3333344]
 [2.0000005 1.0000002 1.0000002]]
Time taken: 0.12 seconds
doubled game: [[4.666668  6.6666684 8.666669 ]
 [4.000001  2.0000005 2.0000005]]
Time taken: 0.03 seconds


p = 0.3: [[1.6899999 2.69      3.69     ]
 [1.9999999 1.        1.       ]]
Time taken: 0.07 seconds
p = 0.7: [[2.89       3.8899999  4.89      ]
 [2.         1.         0.99999994]]
Time taken: 0.00 seconds
